<a href="https://colab.research.google.com/github/nembrinj/ARCHEST_2026/blob/main/colab/3DGaussianSplatting_from_INRIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3D Gaussian splatting from INRIA

This colab notebook installs gaussian-splatting software with all requirements (including appropriate cuda toolkit versions) and starts a training process. It requires GPU compute time.

Intallation time is approx 21-22 minutes (with GPU). But some parts can be skipped if the instance already ran recently.

Then the instance can be used for training gaussian splatting representations. Note that data upload time is not insignificant


# Python downgrading

This is necessary to use the specific repo which was built using python 3.7

This cell takes less than 30sec to compute


In [1]:
!wget -O mini.sh https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
!chmod +x mini.sh
!bash ./mini.sh -b -f -p /usr/local
!conda install -q -y python=3.7
import sys
sys.path.append('/usr/local/lib/python3.7/site-packages')
!python --version  # Should say Python 3.7.x

--2026-08-16 10:00:00--  https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:20f1, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 90665082 (86M) [application/x-sh]
Saving to: ‘mini.sh’

mini.sh             100%[===================>]  86.46M   263MB/s    in 0.3s    

2026-08-16 10:00:00 (263 MB/s) - ‘mini.sh’ saved [90665082/90665082]

PREFIX=/usr/local
Unpacking payload ...
                                                                                              
Installing base environment...





Preparing transaction: - \ | / done
Executing transaction: \ | / - \ | / - \ | / - \ | / - \ | / done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior

# CUDA 11.8

This can takes approx. 14 min to compute

In [2]:
# check if cuda installer already downloaded
!ls -al

total 88564
drwxr-xr-x 1 root root     4096 Aug 16 10:00 .
drwxr-xr-x 1 root root     4096 Aug 16 09:56 ..
drwxr-xr-x 4 root root     4096 Aug 10 13:31 .config
-rwxr-xr-x 1 root root 90665082 Feb 13  2025 mini.sh
drwxr-xr-x 1 root root     4096 Aug 10 13:31 sample_data


In [3]:
# download only if needed (this is the main time consuming part)
!wget https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
!chmod +x cuda_11.8.0_520.61.05_linux.run

--2026-08-16 10:00:35--  https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.15.241.65, 23.15.241.19
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.15.241.65|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4336730777 (4.0G) [application/octet-stream]
Saving to: ‘cuda_11.8.0_520.61.05_linux.run’

cuda_11.8.0_520.61. 100%[===================>]   4.04G  10.0MB/s    in 7m 8s   

2026-08-16 10:07:43 (9.67 MB/s) - ‘cuda_11.8.0_520.61.05_linux.run’ saved [4336730777/4336730777]



In [4]:
# install appropriate cuda version
!./cuda_11.8.0_520.61.05_linux.run --silent --toolkit --no-drm --no-man-page
import os
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia'
!nvcc --version  # Should show CUDA 11.8

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0


#Pytorch with cuda

This cell takes approx 2-3 min to compute.

__As it requires session restart, do not run cells further down before completion__

In [5]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu116
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 509.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 51.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 76.4 MB/s eta 0:00:00
  Obtaining dependency information for pillow!=8.3.*,>=5.3.0 from https://files.pythonhosted.org/packages/2c/a2/2d565cb1d754384a88998b9c86daf803a3a7908577875231eb99b8c7973d/Pillow-9.5.0-cp37-cp37m-manylinux_2_28_x86_64.whl.metadata
Discarding https://files.pythonhosted.org/packages/2c/a2/2d565cb1d754384a88998b9c86daf803a3a7908577875231eb99b8c7973d/Pillow-9.5.0-cp37-cp37m-manylinux_2_28_x86_64.whl#sha256=35f6e77122a0c0762268216315bf239cf52b88865bba522999dc38f1c52b9b47 (from https://download.pytorch.org/whl/cu116/pillow/) (requires-python:>=3.7): Requested pillow!=8.3.*,>=5.3.0 from https://files.pythonhosted.org/packages/2c/a2/2d565cb1d754384a88998b9c86daf803a3a7908577

# Verification of the versions

In [1]:
import torch
print(torch.cuda.is_available())  # Should be True
print(torch.version.cuda)        # Should be 11.3 (from PyTorch)
print(torch.cuda.get_device_name(0))  # Should show GPU

True
12.8
Tesla T4


In [2]:
!nvidia-smi

Sun Aug 16 10:18:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip uninstall diff-gaussian-rasterization simple-knn

In [4]:
# clone the INRIA gaussian-splatting git and intall it
%cd /content
!rm -r gaussian-splatting
!git clone --recursive https://github.com/camenduru/gaussian-splatting

!pip install -q plyfile

%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn


/content
rm: cannot remove 'gaussian-splatting': No such file or directory
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 603, done.
remote: Total 603 (delta 0), reused 0 (delta 0), pack-reused 603 (from 1)
Receiving objects: 100% (603/603), 2.09 MiB | 5.21 MiB/s, done.
Resolving deltas: 100% (346/346), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects: 100% (174/174), done.        
remote: Total 3293 (delta 171), reused 280 

In [9]:
# upload your data
# 1. zip your data on the server
# 2. download it locally, put it on your google drive (new->file upload...) in a "colmap" folder
# 3. run the cell to select the file and unzip it

from google.colab import drive

drive.mount('/content/drive')

# if your zip file is in My Drive/colmap/
data_path = "/content/drive/MyDrive/colmap"

# change to the name of the zip file
zip_name="test"

dir = "/content/gaussian-splatting/"+zip_name
zipfile = data_path+"/"+zip_name+".zip"

!mkdir -p $dir
!unzip -q $zipfile -d $dir


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/gaussian-splatting/test/colmap_data/images/380.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/gaussian-splatting/test/colmap_data/images/1190.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/gaussian-splatting/test/colmap_data/images/1210.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [ ]:
# train the gaussian splats
!python train.py -s /content/gaussian-splatting/test/colmap_data/

Optimizing 
Output folder: ./output/710ae250-7 [16/08 10:49:06]
Tensorboard not available: not logging progress [16/08 10:49:06]
Reading camera 89/89 [16/08 10:49:06]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [16/08 10:49:06]
Loading Training Cameras [16/08 10:49:06]
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly specify '--resolution/-r' as 1 [16/08 10:49:06]
Loading Test Cameras [16/08 10:49:18]
Number of points at initialisation :  8392 [16/08 10:49:18]
Training progress:  23% 7000/30000 [09:48<35:58, 10.66it/s, Loss=0.0218483]
[ITER 7000] Evaluating train: L1 0.011972489021718503 PSNR 35.63180618286133 [16/08 10:59:07]

[ITER 7000] Saving Gaussians [16/08 10:59:07]
Training progress:  39% 11590/30000 [17:22<30:05, 10.19it/s, Loss=0.0199403]

In [ ]:
# use a standard dataset for testing
!wget https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
!unzip tandt_db.zip

!python train.py -s /content/gaussian-splatting/tandt/train